# 01 · Laboratorio 2D — la matriz 3 × 4 con score exacto y aprendido

Notebook único del estudio de ablación sobre la **mezcla de 8 gaussianas**. Reemplaza y absorbe los seis notebooks 2D previos
(dato y forward, entrenamiento y sampleo, reconstrucción desde checkpoint, y los tres de auditoría: marginales, score vs
epsilon, ruido del último paso).

La matriz del estudio son **dos ejes independientes**:

- **Eje 1 — forward SDE** (`vp`, `ve`, `sub_vp`): define cómo se destruye la distribución hacia ruido, y por lo tanto el score
  que hay que aprender. Cambiarlo **obliga a reentrenar**.
- **Eje 2 — sampler del reverso** (`euler`, `pf_ode`, `heun`, `pc`): integra el mismo score de distintas formas. Cambiarlo
  **no** requiere reentrenar.

Eso da **3 × 4 = 12 celdas**, y acá se corren **dos veces**:

| grilla | score | qué aísla |
|---|---|---|
| **A** | **exacto**, del oráculo analítico | toda diferencia entre celdas es del **sampler** (Eje 2 puro) |
| **B** | **aprendido**, red entrenada por DSM | lo que pasa de verdad, con el error de la red incluido |

Comparar A contra B es lo que **separa el error de estimación del score del error de discretización**. Sin el score exacto los
dos vienen mezclados y son indistinguibles: es el desbloqueo concreto que da tener la densidad en forma cerrada.

### Cuatro reglas metodológicas que este notebook sí respeta

Las tres primeras nacieron de leer una corrida anterior y encontrarle los agujeros; están acá para que los números signifiquen
algo:

1. **Presupuesto de NFE igualado.** Heun y el predictor–corrector cuestan **dos** evaluaciones del score por paso; Euler y
   PF-ODE, una. Comparar las cuatro con el mismo `n_steps` le regala el doble de cómputo a dos de ellas. Acá se fija el
   presupuesto en **NFE** y se derivan los pasos de cada sampler.
2. **Piso del estimador medido.** Toda métrica sobre una muestra finita tiene un piso: el valor que da una **muestra perfecta**.
   Se mide explícitamente sampleando de la mezcla exacta, y se dibuja en todos los gráficos. Sin esa línea es imposible saber
   si una diferencia es real o es ruido de Monte Carlo.
3. **Varias semillas.** Cada celda se corre con varias semillas de sampleo, y cada SDE se **reentrena** con varias semillas, así
   que las tablas van con media ± desvío en vez de un número suelto.
4. **Más de una métrica.** Ninguna métrica escalar ve todos los modos de falla; se muestra abajo con contraejemplos medidos.

### El desbloqueo analítico

Si $p_0 = \sum_k w_k\,\mathcal N(\mu_k, \Sigma_k)$ y el kernel forward es gaussiano, entonces $p_t$ **sigue siendo** una mezcla
de gaussianas:

$$p_t = \sum_k w_k\,\mathcal N\!\left(\alpha_t \mu_k,\; \alpha_t^2\Sigma_k + \sigma_t^2 I\right)$$

y por lo tanto hay **score exacto en todo $(x,t)$**:

$$\nabla\log p_t(x) = -\sum_k r_k(x,t)\,\Sigma_k(t)^{-1}\!\left(x - \alpha_t\mu_k\right),\qquad r_k = \text{responsabilidad posterior}$$

> **Device**: el notebook detecta CUDA y enruta a GPU lo que puede aprovecharla —el entrenamiento, el sampleo de las celdas y
> las evaluaciones de score—. Dos cosas quedan en CPU por construcción: la **cuadratura 2D** (utilidad numérica autónoma, sin
> parámetro de device) y la **asignación óptima** de la $W_2$ exacta (solver de SciPy). Ninguna de las dos es el cuello.
>
> **Costo**: dominado por el entrenamiento, `SEEDS_TRAIN × 3` corridas. Con los valores por defecto son ~10 min en GPU; el
> resto del notebook son ~4 min. La celda de setup imprime el presupuesto estimado y todas las perillas están ahí arriba.

In [ ]:
# --- Setup: bootstrap de sys.path + imports + perillas ---
import sys
import pathlib
import time
from collections import namedtuple

_here = pathlib.Path.cwd()
_root = None
for _cand in (_here, *_here.parents):
    if (_cand / "src" / "diffusion").is_dir():
        _root = _cand
        break
if _root is None:
    raise RuntimeError(f"No encontré src/diffusion subiendo desde {_here}")
_src = str((_root / "src").resolve())
if _src not in sys.path:
    sys.path.insert(0, _src)

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment

from diffusion.analytic import MixtureOracle
from diffusion.data_generation import ExactGaussianMixture, infinite_bare
from diffusion.models import ScoreMLP
from diffusion.samplers import available_samplers, make_sampler
from diffusion.sde import make_sde
from diffusion.training import TrainConfig, train

SEED = 1
torch.manual_seed(SEED)
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# --- Device ---------------------------------------------------------------------
# Se detecta una sola vez y se usa en TODO lo que puede aprovecharlo: el
# entrenamiento (TrainConfig.device), el sampleo (sample(device=...) con un
# generator del mismo device) y los tensores que alimentan a la red y al oraculo.
# El score del oraculo es agnostico al device: promueve sus parametros en cada
# llamada al device del estado que recibe.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# En GPU conviene autotunear los matmuls una vez y habilitar TF32: las shapes de
# este laboratorio son fijas en todos los pasos.
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")


def a_device(x):
    """numpy o tensor -> tensor float32 en DEVICE."""
    t = torch.as_tensor(x) if not torch.is_tensor(x) else x
    return t.to(device=DEVICE, dtype=torch.float32)


def gen_device(semilla=SEED):
    """Generator en DEVICE: randn con generator exige que coincidan."""
    g = torch.Generator(device=DEVICE)
    g.manual_seed(semilla)
    return g


# --- Los dos ejes ---------------------------------------------------------------
SDES = ["vp", "ve", "sub_vp"]
SAMPLERS = ["euler", "pf_ode", "heun", "pc"]
T_EPS = 1e-4

# Varianza del prior de cada SDE: es un dato que hay que declarar, porque no se puede
# leer de la SDE de forma exacta y generica. VP y sub-VP parten de N(0, I); VE de
# N(0, sigma_max^2 I), y sigma_max vale 5 por defecto.
PRIOR_VAR = {"vp": 1.0, "sub_vp": 1.0, "ve": 25.0}

# --- Presupuesto de computo: NFE, no pasos --------------------------------------
# Contado sobre el codigo de cada sampler (llamadas a score_fn por paso):
#   euler   -> 1   (_reverse_drift)
#   pf_ode  -> 1   (_pfode_drift)
#   heun    -> 2   (drift en t y en el estado predicho a t+dt)
#   pc      -> 1 + n_corrector  (predictor + cada correccion de Langevin)
# Igualar n_steps entre los cuatro le regala el doble de computo a heun y a pc, asi
# que se fija el NFE y se derivan los pasos.
N_CORRECTOR = 1
NFE_POR_PASO = {"euler": 1, "pf_ode": 1, "heun": 2, "pc": 1 + N_CORRECTOR}
KW_SAMPLER = {"pc": {"n_corrector": N_CORRECTOR}}
NFE = 400


def pasos_para(nom_sampler, nfe=NFE):
    """Pasos de integracion que gastan `nfe` evaluaciones de score con este sampler."""
    return max(1, nfe // NFE_POR_PASO[nom_sampler])


# --- Perillas de costo ----------------------------------------------------------
NUM_STEPS = 50_000     # pasos de entrenamiento por corrida
SEEDS_TRAIN = 3        # redes independientes por SDE (separa "sub-VP es mejor" de "salio bien")
SEEDS_EVAL = 3         # semillas de sampleo por celda; se aparean con las de entrenamiento
SEEDS_PISO = 10        # muestras exactas para medir el piso de cada metrica
N_EVAL = 20_000        # muestras por celda (el piso de la discrepancia baja como 1/sqrt(N))
N_W2 = 1_500           # submuestra para la W2 exacta: la asignacion optima es O(n^3)
N_KSD = 4_000          # submuestra para la KSD: U-estadistico O(n^2)
KSD_BLOQUE = 512       # filas por bloque en la KSD, para acotar la memoria
N_SWEEP = 8_000        # muestras por punto del barrido de NFE
SEEDS_SWEEP = 2
NFES = [50, 100, 200, 400, 800]           # barrido distribucional (los cuatro samplers)
NFES_PATH = [25, 50, 100, 200, 400, 800]  # barrido pathwise (solo deterministas)
NFE_REF = 6_400        # referencia del barrido pathwise
N_PATH = 2_000

print("paquete en:", _root)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(), "| DEVICE:", DEVICE)
print("samplers:", available_samplers())
print()
print(f"presupuesto igualado: NFE={NFE}  ->  " +
      "  ".join(f"{s}:{pasos_para(s)} pasos" for s in SAMPLERS))
print(f"entrenamiento: {SEEDS_TRAIN} semillas x {len(SDES)} SDEs x {NUM_STEPS:,} pasos "
      f"= {SEEDS_TRAIN * len(SDES)} corridas")
if DEVICE == "cpu":
    print()
    print("AVISO: sin GPU visible. El notebook corre igual pero el entrenamiento domina")
    print("el tiempo total. Bajá NUM_STEPS y SEEDS_TRAIN si querés una pasada rápida.")

## 1. El dato: una mezcla de 8 gaussianas de parámetros exactos

Acá está el primer cambio respecto de los notebooks viejos. Ellos usaban la mezcla *legacy*, que es isotrópica, de pesos
parejos y **no publica sus centros**; y encima la estandarizaban con una transformación estimada de la propia muestra, así que
los parámetros efectivos dependían de `n` y de la semilla.

El oráculo necesita $(w_k, \mu_k, \Sigma_k)$ **en forma cerrada**, así que usamos la mezcla exacta: los parámetros son entrada
explícita y quedan consultables. Sin eso no hay score exacto, y por lo tanto no hay grilla A.

> ⚠️ **Detalle que muerde:** `sample()` devuelve los puntos **agrupados por componente** (por eso `color_` sale ordenado). Un
> `x[:m]` no es una submuestra de la mezcla, es un pedazo de uno o dos modos. Todas las submuestras de este notebook salen de
> `submuestra()`, que permuta primero.

In [ ]:
RING = dict(n_components=8, radius=5.0, scale=0.3)
mixtura = ExactGaussianMixture.ring(seed=SEED, **RING)

N_REF = 4000
x_ref = mixtura.sample(N_REF)          # (N, 2) float32
color_ref = mixtura.color_             # componente de cada punto


def muestra_exacta(n, semilla):
    """n muestras de la MISMA mezcla (la geometría no depende de la semilla), desordenadas."""
    x = ExactGaussianMixture.ring(seed=semilla, **RING).sample(n)
    return x[np.random.default_rng(semilla).permutation(len(x))]


def submuestra(x, n, semilla=0):
    """Submuestra aleatoria sin reemplazo (nunca un slice: el orden agrupa por componente)."""
    if len(x) <= n:
        return np.ascontiguousarray(x)
    idx = np.random.default_rng(semilla).choice(len(x), n, replace=False)
    return np.ascontiguousarray(x[idx])


print("pesos     :", np.round(mixtura.weights_, 4))
print("medias[:3]:", np.round(mixtura.means_[:3], 3), "...")
print("covarianza de la componente 0:\n", np.round(mixtura.covariances_[0], 4))
print("color_ ordenado por componente:", bool((np.diff(color_ref) >= 0).all()),
      "-> slicing prohibido, usar submuestra()")

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.scatter(x_ref[:, 0], x_ref[:, 1], c=color_ref, cmap="tab10", s=7, alpha=0.75)
ax.set_aspect("equal")
ax.set_title(f"$p_{{data}}$: mezcla exacta de 8 gaussianas (N={N_REF})")
plt.show()

## 2. El forward: cómo cada SDE destruye el dato

Snapshots de $x_t$ a tiempos crecientes, más el prior del que arranca el reverso. Los límites de los ejes se comparten
**por columna**: a un mismo tiempo las tres SDEs se dibujan en la misma escala, así la comparación es justa, y como las
columnas difieren entre sí el **estado final** de cada SDE queda directamente comparable.

- **VP** y **sub-VP** contraen la media hacia cero y acotan la varianza → prior $\mathcal N(0,I)$.
- **VE** mantiene la media y **explota la varianza** → prior $\mathcal N(0,\sigma_{max}^2 I)$, mucho más ancho.

In [ ]:
times = [T_EPS, 0.25, 0.5, 0.75, 1.0]
col_titles = [f"t={t:.2f}" for t in times] + ["prior $p_T$"]
x0_t = a_device(x_ref)
gen = gen_device()

nubes = []
for nombre in SDES:
    sde = make_sde(nombre)
    fila = []
    for t in times:
        tt = torch.full((x0_t.shape[0],), float(t), device=DEVICE)
        xt, _ = sde.perturb(x0_t, tt, generator=gen)
        fila.append(xt.cpu().numpy())
    fila.append(sde.prior_sampling((x0_t.shape[0], *sde.data_shape),
                                   generator=gen, device=DEVICE).cpu().numpy())
    nubes.append(fila)

# Escala compartida por columna (mismo t => misma escala entre SDEs).
lims = [1.1 * max(float(np.abs(nubes[i][j]).max()) for i in range(len(SDES)))
        for j in range(len(times) + 1)]

fig, axes = plt.subplots(len(SDES), len(times) + 1,
                         figsize=(2.3 * (len(times) + 1), 2.3 * len(SDES)))
for i, nombre in enumerate(SDES):
    for j in range(len(times) + 1):
        ax = axes[i, j]
        nube = nubes[i][j]
        if j < len(times):
            ax.scatter(nube[:, 0], nube[:, 1], c=color_ref, cmap="tab10", s=3, alpha=0.5)
        else:
            ax.scatter(nube[:, 0], nube[:, 1], c="0.45", s=3, alpha=0.5)
        ax.set_xlim(-lims[j], lims[j]); ax.set_ylim(-lims[j], lims[j])
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        if i == 0:
            ax.set_title(col_titles[j], fontsize=10)
    axes[i, 0].set_ylabel(nombre.upper(), fontsize=12)
fig.suptitle("Eje 1 — proceso forward: dato → ruido", y=1.002, fontsize=13)
fig.tight_layout()
plt.show()

## 3. El oráculo: densidad y score exactos

Con la mezcla exacta y una SDE, el oráculo da $p_t$, $\log p_t$ y $\nabla\log p_t$ cerrados. Abajo: la densidad exacta como
contorno y el score exacto como quiver, en dos tiempos. A $t$ chico el campo apunta a los 8 modos; a $t$ grande se aplana
hacia el origen.

In [ ]:
def campo_y_densidad(oraculo, t_val, lim, n=28, n_dens=160):
    """Score (quiver, normalizado) y log-densidad (contorno) exactos en una grilla."""
    xs = np.linspace(-lim, lim, n)
    gx, gy = np.meshgrid(xs, xs)
    pts = a_device(np.stack([gx.ravel(), gy.ravel()], 1))
    tt = torch.full((pts.shape[0],), float(t_val), device=DEVICE)
    s = oraculo.score(pts, tt).cpu().numpy()

    ds = np.linspace(-lim, lim, n_dens)
    dx, dy = np.meshgrid(ds, ds)
    dpts = a_device(np.stack([dx.ravel(), dy.ravel()], 1))
    dtt = torch.full((dpts.shape[0],), float(t_val), device=DEVICE)
    logp = oraculo.log_prob(dpts, dtt).cpu().numpy().reshape(n_dens, n_dens)
    return gx, gy, s, dx, dy, logp


oraculo_vp = MixtureOracle(mixtura, make_sde("vp"))
t_vals = [0.05, 0.6]
fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.4))
for ax, tv in zip(axes, t_vals):
    gx, gy, s, dx, dy, logp = campo_y_densidad(oraculo_vp, tv, lim=7.0)
    ax.contourf(dx, dy, logp, levels=25, cmap="Blues")
    mag = np.linalg.norm(s, axis=1)
    ax.quiver(gx.ravel(), gy.ravel(), s[:, 0] / (mag + 1e-9), s[:, 1] / (mag + 1e-9),
              color="0.25", scale=34, width=0.0035)
    ax.set_aspect("equal")
    ax.set_title(f"VP · $\\log p_t$ (fondo) y $\\nabla\\log p_t$ exacto (flechas), t={tv}")
fig.tight_layout()
plt.show()

### Sesgo de inicialización, por SDE

El sampler arranca del prior de su SDE, pero la distribución real en $t=T$ **no es** exactamente ese prior. Esa diferencia es
un sesgo que se paga antes de dar el primer paso.

Se reportan **dos números** porque la divergencia entre una mezcla y una gaussiana **no tiene forma cerrada** (la entropía
diferencial de una mezcla no la tiene; solo el término cruzado). Así que: una **cota superior** rigurosa de forma cerrada, por
convexidad de la KL, y un **valor de referencia** por cuadratura 2D con su tolerancia. La cota domina al valor, y con una sola
componente coinciden.

In [ ]:
print(f"{'SDE':8s} {'cota':>12s} {'valor':>12s} {'tolerancia':>12s} {'masa':>10s}")
for nombre in SDES:
    ora = MixtureOracle(mixtura, make_sde(nombre))
    rep = ora.initialization_bias(PRIOR_VAR[nombre])
    print(f"{nombre:8s} {rep.bound:12.6f} {rep.value:12.6f} {rep.tolerance:12.2e} {rep.mass:10.6f}")

print()
print("VP y sub-VP contraen la media, así que llegan cerca de su prior N(0,I).")
print("VE solo agranda la varianza: su sesgo es órdenes de magnitud mayor, y solo baja")
print("si sigma_max domina la escala del dato.")

## 4. El banco de métricas, y su piso

Antes de comparar nada hay que saber **cuánto mide una muestra perfecta**. Se usan tres métricas, y la razón de usar tres es
que ninguna sola alcanza: más abajo se mide, con corrupciones controladas, que cada una es ciega a algo.

| métrica | qué es | qué necesita | ciega a |
|---|---|---|---|
| **discrepancia** | peor de tres errores adimensionales sobre las 8 componentes: **peso**, **media** (en unidades del radio RMS) y **covarianza** (Frobenius relativa), asignando cada muestra a su componente por Mahalanobis | los parámetros exactos | estructura dentro de una componente |
| **$W_2$ exacta** | 2-Wasserstein **empírica exacta** entre la nube generada y una muestra exacta de referencia. Con dos nubes de igual tamaño y peso uniforme el plan óptimo es una **permutación** (los vértices del politopo de Birkhoff), así que la asignación óptima —Hungarian— da el valor exacto, sin regularización entrópica | solo muestras | difuminado chico (su piso de muestra finita lo tapa) |
| **KSD²** | *kernel Stein discrepancy* con kernel IMQ, como U-estadístico. **Solo necesita $\nabla\log p_0$** — es la métrica que existe gracias al oráculo | el score exacto | **modos faltantes** (si la masa que queda vive sobre los modos de $p$, la identidad de Stein se sigue cumpliendo casi) |

Dos detalles de implementación que importan:

- La KSD se evalúa contra $\nabla\log p_0$, que se obtiene del oráculo de VP a $t=10^{-6}$ (ahí $\alpha_t\approx 1$ y
  $\sigma_t\approx 3\cdot 10^{-4}$, o sea $10^{-3}$ de la escala de una componente). Se reusa así la implementación ya testeada
  en vez de duplicar la fórmula.
- El U-estadístico de la KSD es **no sesgado**, así que bajo la hipótesis nula fluctúa alrededor de cero y **puede dar
  negativo**. Se reporta con signo: un valor dentro de la banda del piso significa "indistinguible de exacto".

In [ ]:
# --- Las tres metricas -----------------------------------------------------------
Metricas = namedtuple("Metricas", "peor peso media cov w2 ksd2")

T_P0 = 1e-6          # a este t, el oraculo de VP es p_0 (alpha~1, sigma~3e-4)
C_IMQ = 1.0          # ancho del kernel IMQ, calibrado abajo
EXP_IMQ = -0.5       # beta del kernel (c^2 + r^2)^beta
_ORA_P0 = MixtureOracle(mixtura, make_sde("vp"))


def score_p0(x):
    """Score exacto de p_0, via el oraculo a t=T_P0."""
    return _ORA_P0.score(x, torch.full((x.shape[0],), T_P0, device=x.device))


def errores_por_componente(muestras, mix=None):
    """(peso, media, cov): peor error adimensional sobre las K componentes."""
    mix = mix if mix is not None else mixtura
    x = np.asarray(muestras, dtype=np.float64)
    w, mu, cov = mix.weights_, mix.means_, mix.covariances_
    inv = np.linalg.inv(cov)
    d = x[:, None, :] - mu[None]                                  # (N,K,2)
    m2 = np.einsum("nki,kij,nkj->nk", d, inv, d)                  # Mahalanobis^2
    asign = m2.argmin(1)
    peor_w = peor_mu = peor_cov = 0.0
    for k in range(len(w)):
        sel = x[asign == k]
        peor_w = max(peor_w, abs(len(sel) / len(x) - w[k]))
        if len(sel) < 5:                                          # componente vacia
            peor_mu, peor_cov = max(peor_mu, 1.0), max(peor_cov, 1.0)
            continue
        radio = np.sqrt(np.trace(cov[k]))                         # radio RMS
        peor_mu = max(peor_mu, np.linalg.norm(sel.mean(0) - mu[k]) / radio)
        c = np.cov(sel.T)
        peor_cov = max(peor_cov, np.linalg.norm(c - cov[k]) / np.linalg.norm(cov[k]))
    return peor_w, peor_mu, peor_cov


def w2_exacta(x, y):
    """W2 empirica EXACTA entre dos nubes del mismo tamano, por asignacion optima."""
    x = np.asarray(x, dtype=np.float64); y = np.asarray(y, dtype=np.float64)
    if len(x) != len(y):
        raise ValueError(f"la asignacion exacta pide nubes iguales; {len(x)} vs {len(y)}")
    C = ((x[:, None, :] - y[None]) ** 2).sum(-1)
    fila, col = linear_sum_assignment(C)
    return float(np.sqrt(C[fila, col].mean()))


def ksd2(muestras, semilla=0):
    """KSD^2 con kernel IMQ como U-estadistico (no sesgado: puede ser negativo).

    Se acumula por bloques de filas para acotar la memoria, y en float64 porque los
    terminos individuales son O(10^3) y se suman ~n^2 de ellos.
    """
    x = a_device(submuestra(muestras, N_KSD, semilla))
    n, d = x.shape
    b = EXP_IMQ
    with torch.no_grad():
        s_all = score_p0(x).double()
        xd = x.double()
        total = 0.0
        for i0 in range(0, n, KSD_BLOQUE):
            i1 = min(i0 + KSD_BLOQUE, n)
            si = s_all[i0:i1]
            r = xd[i0:i1, None, :] - xd[None]                     # (m,n,d)
            r2 = (r * r).sum(-1)
            base = C_IMQ ** 2 + r2
            k = base ** b
            kb1 = base ** (b - 1.0)
            kb2 = base ** (b - 2.0)
            u = ((si @ s_all.T) * k
                 - 2.0 * b * kb1 * (si[:, None, :] * r).sum(-1)
                 + 2.0 * b * kb1 * (s_all[None, :, :] * r).sum(-1)
                 - 4.0 * b * (b - 1.0) * kb2 * r2
                 - 2.0 * b * d * kb1)
            idx = torch.arange(i0, i1, device=u.device)
            u[idx - i0, idx] = 0.0                                # anular la diagonal
            total += float(u.sum())
    return total / (n * (n - 1))


X_REF_W2 = muestra_exacta(N_W2, 9_999)     # referencia fija: todas las celdas contra la misma


def metricas(muestras, con_w2=True, con_ksd=True, semilla=0):
    """Las tres metricas de una nube."""
    pw, pm, pc = errores_por_componente(muestras)
    w = w2_exacta(submuestra(muestras, N_W2, semilla), X_REF_W2) if con_w2 else np.nan
    k = ksd2(muestras, semilla) if con_ksd else np.nan
    return Metricas(max(pw, pm, pc), pw, pm, pc, w, k)


def agrega(lista):
    """[Metricas] -> {campo: (media, desvio)}."""
    return {f: (float(np.mean([getattr(m, f) for m in lista])),
                float(np.std([getattr(m, f) for m in lista])))
            for f in Metricas._fields}


print("banco de métricas listo")
print(f"  discrepancia sobre N={N_EVAL}, W2 exacta sobre n={N_W2}, KSD sobre n={N_KSD}")

In [ ]:
# --- El piso: que mide una muestra PERFECTA -------------------------------------
t0 = time.time()
piso_m = [metricas(muestra_exacta(N_EVAL, 300 + s), semilla=s) for s in range(SEEDS_PISO)]
PISO = agrega(piso_m)

print(f"PISO del estimador — {SEEDS_PISO} muestras exactas de la mezcla, N={N_EVAL}")
print(f"{'métrica':14s} {'media':>11s} {'desvío':>10s} {'|máx|':>11s}")
for campo in ("peor", "peso", "media", "cov", "w2", "ksd2"):
    vals = np.array([getattr(m, campo) for m in piso_m])
    print(f"{campo:14s} {PISO[campo][0]:11.4g} {PISO[campo][1]:10.3g} {np.abs(vals).max():11.4g}")

# Banda del piso: media + 2 desvios. Cualquier celda por debajo es indistinguible
# de una muestra perfecta, y su valor NO se puede interpretar como error del sampler.
BANDA = {c: PISO[c][0] + 2 * PISO[c][1] for c in Metricas._fields}
print()
print("banda (media + 2σ) — por debajo de esto no hay señal:",
      {c: round(BANDA[c], 4) for c in ("peor", "w2", "ksd2")})
print(f"({time.time()-t0:.1f}s)")

### Las tres métricas son complementarias (medido, no argumentado)

Se corrompe una muestra exacta de cinco formas distintas y se mira qué detecta cada métrica. El punto es que **hay una
corrupción para la que cada métrica queda dentro de su piso**, así que reportar una sola sería engañoso.

In [ ]:
x_ok = muestra_exacta(N_EVAL, 100)
_rng = np.random.default_rng(0)
_lejos = np.linalg.norm(x_ok - mixtura.means_[0], axis=1) > 1.0

corrupciones = {
    "exacta (control)": x_ok,
    "difuminada 0.10": x_ok + _rng.normal(0, 0.10, x_ok.shape),
    "difuminada 0.30": x_ok + _rng.normal(0, 0.30, x_ok.shape),
    "escalada 1.02": x_ok * 1.02,
    "1 modo perdido": x_ok[_lejos],
    "pesos sesgados": np.concatenate([x_ok[x_ok[:, 0] > 0]] * 2)[:N_EVAL],
}

print(f"{'corrupción':18s} {'peor':>9s} {'peso':>8s} {'media':>8s} {'cov':>8s} {'W2':>8s} {'KSD²':>11s}")
print(f"{'PISO (media+2σ)':18s} {BANDA['peor']:9.4f} {BANDA['peso']:8.4f} {BANDA['media']:8.4f} "
      f"{BANDA['cov']:8.4f} {BANDA['w2']:8.4f} {BANDA['ksd2']:11.3e}")
print("-" * 76)
for nombre, xc in corrupciones.items():
    m = metricas(np.ascontiguousarray(xc.astype(np.float32)))
    marca = lambda v, c: "*" if v > BANDA[c] else " "
    print(f"{nombre:18s} {m.peor:9.4f}{marca(m.peor,'peor')}{m.peso:7.4f}{marca(m.peso,'peso')}"
          f"{m.media:7.4f}{marca(m.media,'media')}{m.cov:7.4f}{marca(m.cov,'cov')}"
          f"{m.w2:7.4f}{marca(m.w2,'w2')}{m.ksd2:10.3e}{marca(m.ksd2,'ksd2')}")
print()
print("* = por encima del piso (detectado).  Leer las columnas sin asterisco:")
print("  · la KSD no ve un modo perdido: la masa que sobra sigue sobre modos de p,")
print("    así que la identidad de Stein se sigue cumpliendo casi.")
print("  · la W2 no ve difuminado chico: su piso de muestra finita lo tapa.")
print("  · el término 'cov' de la discrepancia sí ve difuminado — es el que mide truncación.")

## 5. Entrenar las tres SDEs, con varias semillas

Cambiar el forward SDE **obliga a reentrenar** (cada SDE define una $p_t$ distinta y por lo tanto otro score). La red es la
**variable de control**: misma arquitectura y mismos hiperparámetros en todas las corridas, así toda diferencia se atribuye a
la matemática y no a la ingeniería.

Se entrenan `SEEDS_TRAIN` redes **independientes por SDE**. Sin eso no se puede distinguir "esta SDE es mejor" de "esta
corrida salió mejor", que es una confusión fácil de cometer con una sola semilla.

> ⚠️ Las pérdidas finales **no son comparables entre SDEs**: el target y el peso del DSM son distintos en cada una. Sirven para
> ver que cada corrida convergió, no para rankear.

In [ ]:
redes = {nombre: [] for nombre in SDES}       # nombre -> [red_semilla0, red_semilla1, ...]
historias = {nombre: [] for nombre in SDES}
t_inicio = time.time()
for nombre in SDES:
    sde = make_sde(nombre)
    for s in range(SEEDS_TRAIN):
        torch.manual_seed(1_000 + s)             # inicialización distinta por semilla
        red = ScoreMLP(data_dim=2)               # misma arquitectura en todas las corridas
        datos = infinite_bare(mixtura.dataloader(5000, 256, shuffle=True))
        cfg = TrainConfig(num_steps=NUM_STEPS, lr=1e-3, t_eps=T_EPS, seed=s,
                          log_every=NUM_STEPS // 4, device=DEVICE)
        t0 = time.time()
        res = train(sde, red, datos, cfg)
        res.net.eval()
        redes[nombre].append(res.net)
        historias[nombre].append(res.history)
        print(f"  {nombre:7s} semilla {s}  {time.time()-t0:6.1f}s  "
              f"loss {res.history[0]:.4f} -> {res.history[-1]:.4f}")
print(f"total: {(time.time()-t_inicio)/60:.1f} min")

fig, ax = plt.subplots(figsize=(7.5, 4))
for i, nombre in enumerate(SDES):
    for s, h in enumerate(historias[nombre]):
        ax.plot(h, lw=1.0, color=f"C{i}", alpha=0.8,
                label=nombre.upper() if s == 0 else None)
ax.set_yscale("log"); ax.set_xlabel("registro"); ax.set_ylabel("pérdida DSM (log)")
ax.set_title(f"Entrenamiento por DSM · {SEEDS_TRAIN} semillas por SDE (no comparables entre SDEs)")
ax.legend()
plt.show()

print()
print(f"{'SDE':8s} {'loss final (media ± desvío entre semillas)':>44s}")
for nombre in SDES:
    fin = np.array([h[-1] for h in historias[nombre]])
    print(f"{nombre:8s} {fin.mean():>32.4f} ± {fin.std():.4f}")

## 6. ¿Dónde vive el error del score?

Ahora que hay score exacto, el error de la red se puede **medir** en vez de suponer: $\mathbb E_{p_t}\|s_\theta - \nabla\log p_t\|^2$
en función de $t$, con la banda entre semillas de entrenamiento.

Se muestran **dos** curvas porque el absoluto miente: el score verdadero explota como $1/\sigma_t^2$ cuando $t\to 0$, así que un
error absoluto grande ahí puede ser un error **relativo** chiquito. La lectura honesta es la relativa.

In [ ]:
ts = np.geomspace(T_EPS, 1.0, 22)
N_ERR = 4000
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for i, nombre in enumerate(SDES):
    sde = make_sde(nombre)
    ora = MixtureOracle(mixtura, sde)
    x0 = a_device(muestra_exacta(N_ERR, 77))
    curvas_abs, curvas_rel = [], []
    for red in redes[nombre]:
        g = gen_device(5)
        abs_err, rel_err = [], []
        for t in ts:
            tt = torch.full((N_ERR,), float(t), device=DEVICE)
            xt, _ = sde.perturb(x0, tt, generator=g)          # muestras de p_t
            with torch.no_grad():
                s_red = red(xt, tt)
            s_exacto = ora.score(xt, tt)
            num = (s_red - s_exacto).pow(2).sum(1)
            den = s_exacto.pow(2).sum(1)
            abs_err.append(float(num.mean()))
            rel_err.append(float((num / den.clamp_min(1e-30)).mean()))
        curvas_abs.append(abs_err); curvas_rel.append(rel_err)
    for ax, curvas in zip(axes, (np.array(curvas_abs), np.array(curvas_rel))):
        med = curvas.mean(0)
        ax.plot(ts, med, marker="o", ms=3, color=f"C{i}", label=nombre.upper())
        ax.fill_between(ts, curvas.min(0), curvas.max(0), color=f"C{i}", alpha=0.18)

for ax, ttl, yl in zip(axes,
                       ["Error absoluto del score", "Error relativo del score"],
                       [r"$E\,\|s_\theta-\nabla\log p_t\|^2$", "error relativo"]):
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("t"); ax.set_ylabel(yl); ax.set_title(ttl); ax.legend()
fig.suptitle("Banda = mín/máx entre semillas de entrenamiento", y=1.02, fontsize=10)
fig.tight_layout()
plt.show()

## 7. La grilla 3 × 4, a NFE igualado

Las 12 celdas, con el **mismo $x_T$** en todas (para cada semilla de evaluación el prior se sortea del mismo estado, así la
comparación entre celdas es apareada), primero con score exacto y después con la red.

**El presupuesto se iguala en NFE, no en pasos.** Contando las llamadas a `score_fn` en el código de cada sampler:

| sampler | NFE / paso | pasos con NFE = 400 |
|---|---|---|
| `euler` | 1 | 400 |
| `pf_ode` | 1 | 400 |
| `heun` | 2 (drift en $t$ y en el predicho a $t+dt$) | 200 |
| `pc` | 1 + `n_corrector` | 200 |

Igualar `n_steps` —lo natural y lo que se hace sin pensarlo— le da a Heun y a PC **el doble de cómputo** que a Euler y PF-ODE.
El `CLAUDE.md` del proyecto ya pedía "presupuestos de NFE igualados" para la Fase 2; acá se cumple también en 2D.

Cada celda se corre con `SEEDS_EVAL` semillas. En la grilla B la semilla de sampleo se **aparea con la red** de la misma
semilla, así que el desvío que se reporta incluye la variabilidad del entrenamiento, no solo la del sampler.

In [ ]:
def corre_grilla(score_de, titulo, nfe=NFE, n=N_EVAL, semillas=None):
    """Samplea las 12 celdas con varias semillas y NFE igualado."""
    semillas = range(SEEDS_EVAL) if semillas is None else semillas
    out = {}
    fig, axes = plt.subplots(len(SDES), len(SAMPLERS),
                             figsize=(3.0 * len(SAMPLERS), 3.0 * len(SDES)))
    for i, nom_sde in enumerate(SDES):
        sde = make_sde(nom_sde)
        for j, nom_s in enumerate(SAMPLERS):
            pasos = pasos_para(nom_s, nfe)
            ms, ultima = [], None
            for s in semillas:
                smp = make_sampler(nom_s, sde, score_de(nom_sde, sde, s),
                                   n_steps=pasos, t_eps=T_EPS, **KW_SAMPLER.get(nom_s, {}))
                g = gen_device(1_000 + s)          # mismo x_T en toda la grilla
                xs = smp.sample(n, generator=g, device=DEVICE).cpu().numpy()
                ms.append(metricas(xs, semilla=s))
                ultima = xs
            out[(nom_sde, nom_s)] = agrega(ms)
            med, dev = out[(nom_sde, nom_s)]["peor"]
            ax = axes[i, j]
            ax.scatter(x_ref[:, 0], x_ref[:, 1], c="0.85", s=3, alpha=0.5, zorder=0)
            ax.scatter(ultima[:, 0], ultima[:, 1], c="C3", s=2, alpha=0.35, zorder=1)
            ax.set_xlim(-8, 8); ax.set_ylim(-8, 8); ax.set_aspect("equal")
            ax.set_xticks([]); ax.set_yticks([])
            if i == 0:
                ax.set_title(f"{nom_s}\n{pasos} pasos · {nfe} NFE", fontsize=10)
            dentro = med <= BANDA["peor"]
            ax.text(0.03, 0.03, f"{med:.3f}±{dev:.3f}" + ("  (=piso)" if dentro else ""),
                    transform=ax.transAxes, fontsize=8,
                    color="green" if dentro else "darkblue")
        axes[i, 0].set_ylabel(nom_sde.upper(), fontsize=12)
    fig.suptitle(f"{titulo}\npiso del estimador: {BANDA['peor']:.3f} — verde = indistinguible de perfecto",
                 y=1.01, fontsize=12)
    fig.tight_layout()
    plt.show()
    return out


print("Grilla A: score EXACTO (toda diferencia entre celdas es del sampler)")
t0 = time.time()
grilla_exacta = corre_grilla(
    lambda nombre, sde, s: MixtureOracle(mixtura, sde),
    "Grilla A — score exacto del oráculo (filas: forward SDE · columnas: sampler)",
)
print(f"  {time.time()-t0:.1f}s")

In [ ]:
print("Grilla B: score APRENDIDO (el error de la red entra en juego)")
t0 = time.time()
grilla_red = corre_grilla(
    lambda nombre, sde, s: redes[nombre][s % SEEDS_TRAIN],
    "Grilla B — score aprendido por DSM (filas: forward SDE · columnas: sampler)",
)
print(f"  {time.time()-t0:.1f}s")

## 8. La descomposición del error

Las dos grillas lado a lado, como números. Con score exacto lo que queda es **discretización + truncación**; con score
aprendido se suma el **error de estimación del score**. La diferencia entre columnas es, celda por celda, lo que aporta la red.

La columna `piso?` marca las celdas cuyo valor cae dentro de la banda del estimador: ahí el número **no es** el error del
sampler, es ruido de Monte Carlo, y no se puede rankear.

In [ ]:
enc = (f"{'celda':16s} {'A: exacto':>16s} {'B: aprendido':>16s} {'B - A':>8s} "
       f"{'piso?':>7s} {'W2 A':>7s} {'W2 B':>7s} {'KSD² A':>10s} {'KSD² B':>10s}")
print(enc)
print("-" * len(enc))
filas = []
for nom_sde in SDES:
    for nom_s in SAMPLERS:
        a, b = grilla_exacta[(nom_sde, nom_s)], grilla_red[(nom_sde, nom_s)]
        filas.append((f"{nom_sde}/{nom_s}", a, b))
        pa = "A" if a["peor"][0] <= BANDA["peor"] else ""
        pb = "B" if b["peor"][0] <= BANDA["peor"] else ""
        print(f"{nom_sde+'/'+nom_s:16s} "
              f"{a['peor'][0]:8.4f}±{a['peor'][1]:6.4f} "
              f"{b['peor'][0]:8.4f}±{b['peor'][1]:6.4f} "
              f"{b['peor'][0]-a['peor'][0]:+8.4f} {pa+pb:>7s} "
              f"{a['w2'][0]:7.3f} {b['w2'][0]:7.3f} "
              f"{a['ksd2'][0]:10.2e} {b['ksd2'][0]:10.2e}")

print()
print(f"piso (media+2σ): peor={BANDA['peor']:.4f}  W2={BANDA['w2']:.3f}  KSD²={BANDA['ksd2']:.2e}")
print()
print("Promedios (grilla B, la realista) — cuidado: promediar sobre celdas que están")
print("en el piso baja el promedio sin que eso signifique calidad.")
print(f"  por SDE     : " + "  ".join(
    f"{s}={np.mean([grilla_red[(s,k)]['peor'][0] for k in SAMPLERS]):.4f}" for s in SDES))
print(f"  por sampler : " + "  ".join(
    f"{k}={np.mean([grilla_red[(s,k)]['peor'][0] for s in SDES]):.4f}" for k in SAMPLERS))

# --- barras con el piso dibujado ---
etiquetas = [f[0] for f in filas]
va = np.array([f[1]["peor"][0] for f in filas]); ea = np.array([f[1]["peor"][1] for f in filas])
vb = np.array([f[2]["peor"][0] for f in filas]); eb = np.array([f[2]["peor"][1] for f in filas])
y = np.arange(len(filas))
fig, ax = plt.subplots(figsize=(9, 6.5))
ax.barh(y - 0.2, va, xerr=ea, height=0.4, capsize=2,
        label="A · score exacto (discretización + truncación)")
ax.barh(y + 0.2, vb, xerr=eb, height=0.4, capsize=2,
        label="B · score aprendido (+ error de la red)")
ax.axvline(BANDA["peor"], color="k", ls="--", lw=1.2,
           label=f"piso del estimador ({BANDA['peor']:.3f})")
ax.axvspan(0, BANDA["peor"], color="0.85", alpha=0.6, zorder=0)
ax.set_yticks(y); ax.set_yticklabels(etiquetas, fontsize=9); ax.invert_yaxis()
ax.set_xlabel("discrepancia contra la mezcla (menor es mejor)")
ax.set_title(f"Descomposición del error por celda · NFE={NFE} igualado · {SEEDS_EVAL} semillas")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

### Qué término del error se rompe

La discrepancia es el **peor** de tres errores, así que el escalar esconde el diagnóstico. Separados, dicen qué falla:
**peso** = masa mal repartida entre modos (el sampler enruta mal), **media** = modos corridos de lugar, **cov** = modos
demasiado anchos o angostos (difuminado, típicamente truncación).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5.2), sharey=True)
for ax, campo, ttl in zip(axes, ("peso", "media", "cov"),
                          ("peso — masa mal repartida",
                           "media — modos corridos",
                           "cov — modos anchos/angostos")):
    va = np.array([f[1][campo][0] for f in filas])
    vb = np.array([f[2][campo][0] for f in filas])
    ax.barh(y - 0.2, va, height=0.4, label="A · exacto")
    ax.barh(y + 0.2, vb, height=0.4, label="B · aprendido")
    ax.axvline(BANDA[campo], color="k", ls="--", lw=1.1, label="piso")
    ax.axvspan(0, BANDA[campo], color="0.85", alpha=0.6, zorder=0)
    ax.set_title(ttl, fontsize=11); ax.set_xlabel("error adimensional")
axes[0].set_yticks(y); axes[0].set_yticklabels(etiquetas, fontsize=9)
axes[0].invert_yaxis(); axes[0].legend(loc="lower right", fontsize=9)
fig.suptitle("Los tres términos por separado — el escalar 'peor' es el máximo de estos", y=1.02)
fig.tight_layout()
plt.show()

print("Celdas donde el término dominante NO es la covarianza (o sea: no es difuminado):")
for etq, a, b in filas:
    for nom, dd in (("A", a), ("B", b)):
        trio = {c: dd[c][0] for c in ("peso", "media", "cov")}
        dom = max(trio, key=trio.get)
        if dom != "cov" and trio[dom] > BANDA[dom]:
            print(f"  {etq:16s} grilla {nom}: domina '{dom}' = {trio[dom]:.4f} "
                  f"(peso {trio['peso']:.4f} media {trio['media']:.4f} cov {trio['cov']:.4f})")

## 9. Barrido de NFE: calidad por unidad de cómputo

Igualar el NFE en un punto contesta "¿quién gana con este presupuesto?". La pregunta más útil es **cómo escala cada sampler**,
porque el argumento entero de Heun es *mejor calidad por NFE* y el 2º orden solo se paga a NFE grande: puede perder abajo y
ganar arriba. Una sola comparación puntual no distingue esos dos regímenes.

Se barre el presupuesto y se mide con $W_2$ (que es la métrica que ve modos faltantes, el modo de falla que domina cuando hay
poco cómputo). La línea punteada es el piso; una curva que la toca ya no puede mejorar.

In [ ]:
def barrido_nfe(score_de, n=N_SWEEP, semillas=None):
    """{(sde,sampler): {nfe: (media, desvio)}} de W2, barriendo el presupuesto."""
    semillas = range(SEEDS_SWEEP) if semillas is None else semillas
    out = {}
    for nom_sde in SDES:
        sde = make_sde(nom_sde)
        for nom_s in SAMPLERS:
            por_nfe = {}
            for nfe in NFES:
                vals = []
                for s in semillas:
                    smp = make_sampler(nom_s, sde, score_de(nom_sde, sde, s),
                                       n_steps=pasos_para(nom_s, nfe), t_eps=T_EPS,
                                       **KW_SAMPLER.get(nom_s, {}))
                    xs = smp.sample(n, generator=gen_device(1_000 + s),
                                    device=DEVICE).cpu().numpy()
                    vals.append(w2_exacta(submuestra(xs, N_W2, s), X_REF_W2))
                por_nfe[nfe] = (float(np.mean(vals)), float(np.std(vals)))
            out[(nom_sde, nom_s)] = por_nfe
    return out


t0 = time.time()
sweep = {
    "A · score exacto": barrido_nfe(lambda nombre, sde, s: MixtureOracle(mixtura, sde)),
    "B · score aprendido": barrido_nfe(lambda nombre, sde, s: redes[nombre][s % SEEDS_TRAIN]),
}
print(f"barrido: {(time.time()-t0)/60:.1f} min")

fig, axes = plt.subplots(2, len(SDES), figsize=(4.6 * len(SDES), 8.4), sharex=True, sharey=True)
for fi, (titulo, datos) in enumerate(sweep.items()):
    for ci, nom_sde in enumerate(SDES):
        ax = axes[fi, ci]
        for nom_s in SAMPLERS:
            xs = np.array(NFES, dtype=float)
            ys = np.array([datos[(nom_sde, nom_s)][k][0] for k in NFES])
            es = np.array([datos[(nom_sde, nom_s)][k][1] for k in NFES])
            ax.errorbar(xs, ys, yerr=es, marker="o", ms=4, capsize=2, label=nom_s)
        ax.axhline(BANDA["w2"], color="k", ls="--", lw=1.1)
        ax.set_xscale("log"); ax.set_yscale("log")
        if fi == 1:
            ax.set_xlabel("NFE (evaluaciones de score)")
        if ci == 0:
            ax.set_ylabel(f"{titulo}\n$W_2$ exacta")
        ax.set_title(nom_sde.upper(), fontsize=11)
axes[0, 0].legend(fontsize=9)
fig.suptitle("Calidad por cómputo · punteada = piso del estimador (por debajo no hay señal)", y=1.0)
fig.tight_layout()
plt.show()

print(f"{'celda':16s} " + " ".join(f"{'NFE '+str(k):>9s}" for k in NFES))
for titulo, datos in sweep.items():
    print(f"--- {titulo}")
    for nom_sde in SDES:
        for nom_s in SAMPLERS:
            fila = " ".join(f"{datos[(nom_sde,nom_s)][k][0]:9.3f}" for k in NFES)
            print(f"{nom_sde+'/'+nom_s:16s} {fila}")

## 10. Orden de convergencia, sin piso de Monte Carlo

El barrido de arriba mide convergencia **distribucional**, y por eso choca contra el piso del estimador antes de mostrar el
orden del integrador. Para medir el orden hay una forma sin piso: comparar **partícula por partícula** contra una solución de
referencia muy fina, partiendo del **mismo $x_T$**. Como la comparación es apareada, no hay ruido de muestreo — el error
tiende a cero con el paso.

Esto **solo aplica a los samplers deterministas** (`pf_ode`, `heun`). Para `euler` y `pc` cambiar `n_steps` cambia la cantidad
de incrementos brownianos, así que las trayectorias no son comparables punto a punto: haría falta poder **inyectar el camino
browniano** en el sampler, que es justamente una de las extensiones pendientes del módulo. Es la razón concreta por la que esa
tarea existe.

Esperado: pendiente $\approx -1$ para el Euler de la PF-ODE y $\approx -2$ para Heun.

In [ ]:
DETERMINISTAS = ["pf_ode", "heun"]
fig, axes = plt.subplots(1, len(SDES), figsize=(4.6 * len(SDES), 4.2), sharey=True)
pendientes = {}
for ci, nom_sde in enumerate(SDES):
    sde = make_sde(nom_sde)
    ora = MixtureOracle(mixtura, sde)
    # Referencia comun: Heun con NFE_REF. Los dos samplers integran la MISMA PF-ODE,
    # asi que convergen al mismo limite y la referencia sirve para ambos.
    ref = make_sampler("heun", sde, ora, n_steps=pasos_para("heun", NFE_REF), t_eps=T_EPS
                       ).sample(N_PATH, generator=gen_device(1_000), device=DEVICE)
    ax = axes[ci]
    curvas = {}
    for nom_s in DETERMINISTAS:
        errs = []
        for nfe in NFES_PATH:
            xs = make_sampler(nom_s, sde, ora, n_steps=pasos_para(nom_s, nfe), t_eps=T_EPS
                              ).sample(N_PATH, generator=gen_device(1_000), device=DEVICE)
            errs.append(float((xs - ref).pow(2).sum(1).mean().sqrt()))
        errs = np.array(errs)
        curvas[nom_s] = errs
        # Pendiente sobre los presupuestos chicos: los grandes se acercan a la referencia
        # y ahi lo que se mide es la resolucion de la referencia, no el integrador.
        usable = np.array(NFES_PATH) <= NFE_REF // 16
        p = np.polyfit(np.log(np.array(NFES_PATH)[usable]), np.log(errs[usable]), 1)[0]
        pendientes[(nom_sde, nom_s)] = p
        ax.plot(NFES_PATH, errs, marker="o", ms=4, label=f"{nom_s} (pendiente {p:.2f})")
    base = np.array(NFES_PATH, dtype=float)
    ancla = curvas[DETERMINISTAS[0]][0]                 # anclar las guias en el 1er punto
    for orden, est in ((1, ":"), (2, "--")):
        ax.plot(base, ancla * (base / base[0]) ** (-orden), est, color="0.5", lw=1,
                label=f"orden {orden}" if ci == 0 else None)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("NFE"); ax.set_title(nom_sde.upper(), fontsize=11); ax.legend(fontsize=8)
axes[0].set_ylabel(r"RMS $\|x_{NFE}-x_{ref}\|$  (mismo $x_T$)")
fig.suptitle(f"Error de discretización pathwise, score exacto · referencia: Heun con {NFE_REF} NFE", y=1.02)
fig.tight_layout()
plt.show()

print("Pendientes ajustadas (esperado: -1 para pf_ode, -2 para heun):")
for (nom_sde, nom_s), p in pendientes.items():
    print(f"  {nom_sde:8s} {nom_s:8s} {p:+.2f}")

## 11. Truncación: qué queda del último paso

El reverso se integra de $T$ hasta $t_\epsilon > 0$, no hasta cero. Esa truncación deja un **suavizado residual**: lo generado
es $p_{t_\epsilon}$, no $p_0$. Con score exacto se puede aislar el efecto, porque no hay error de red que lo tape.

Es el caso donde se ve para qué sirve haber separado los tres términos: el difuminado entra por **`cov`**, y la $W_2$ y la KSD
son casi ciegas a él (medido en la sección 4).

In [ ]:
eps_vals = [1e-1, 3e-2, 1e-2, 3e-3, 1e-3, 1e-4]
sde = make_sde("vp")
ora = MixtureOracle(mixtura, sde)
res_trunc, sigmas = [], []
for eps in eps_vals:
    smp = make_sampler("pf_ode", sde, ora, n_steps=pasos_para("pf_ode", NFE), t_eps=eps)
    xs = smp.sample(N_EVAL, generator=gen_device(1_000), device=DEVICE).cpu().numpy()
    res_trunc.append(metricas(xs))
    _, sig = ora.marginal_params(torch.tensor([eps]))
    sigmas.append(float(sig))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for campo, mk in (("cov", "o"), ("media", "s"), ("peso", "^")):
    axes[0].plot(sigmas, [getattr(m, campo) for m in res_trunc], marker=mk, label=campo)
    axes[0].axhline(BANDA[campo], color="k", ls=":", lw=0.8)
axes[0].set_xscale("log"); axes[0].set_yscale("log")
axes[0].set_xlabel(r"$\sigma_{t_\epsilon}$ (escala del suavizado residual)")
axes[0].set_ylabel("error adimensional")
axes[0].set_title("Los tres términos: el difuminado entra por 'cov'")
axes[0].legend()

axes[1].plot(sigmas, [m.w2 for m in res_trunc], marker="o", label="$W_2$ exacta")
axes[1].axhline(BANDA["w2"], color="C0", ls="--", lw=1, label="piso $W_2$")
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_xlabel(r"$\sigma_{t_\epsilon}$"); axes[1].set_ylabel("$W_2$")
axes[1].set_title("La $W_2$ casi no lo ve: su piso lo tapa")
axes[1].legend()
fig.tight_layout()
plt.show()

print(f"{'t_eps':>8s} {'sigma':>9s} {'peor':>9s} {'cov':>9s} {'W2':>8s} {'KSD²':>11s}")
for eps, sig, m in zip(eps_vals, sigmas, res_trunc):
    print(f"{eps:8.0e} {sig:9.5f} {m.peor:9.4f} {m.cov:9.4f} {m.w2:8.3f} {m.ksd2:11.2e}")
print(f"{'piso':>8s} {'':9s} {BANDA['peor']:9.4f} {BANDA['cov']:9.4f} "
      f"{BANDA['w2']:8.3f} {BANDA['ksd2']:11.2e}")

## Cierre

Qué quedó medido, y con qué garantías:

- La **matriz 3 × 4** corrió dos veces sobre la misma mezcla, con el mismo $x_T$ en las 12 celdas, la red fija como variable
  de control, **NFE igualado** entre los cuatro samplers y varias semillas de sampleo y de entrenamiento.
- Con **score exacto** toda diferencia entre celdas es del sampler: es el Eje 2 aislado, imposible de ver cuando el error de la
  red está mezclado. La **diferencia entre las dos grillas** es lo que aporta el error de estimación del score, celda por celda.
- El **piso del estimador** está medido y dibujado en todos los gráficos. Es la diferencia entre "este sampler es mejor" y
  "esta muestra tuvo suerte": las celdas marcadas en verde no admiten ranking.
- Se usan **tres métricas** porque se midió que cada una es ciega a un modo de falla distinto: la KSD no ve modos faltantes, la
  $W_2$ no ve difuminado chico, y la discrepancia por componente no ve estructura interna a un modo.
- El **barrido de NFE** separa el régimen de poco cómputo del de mucho, que es donde el 2º orden de Heun puede pagar; y el
  barrido **pathwise** mide el orden del integrador sin piso de Monte Carlo.
- El **sesgo de inicialización** quedó cuantificado por SDE, con cota cerrada y valor por cuadratura, en vez de argumentado.
- La **truncación** se midió contra la escala del suavizado residual que deja, y se ve por qué término entra.

Lo que sigue en el laboratorio:

- El barrido pathwise **no cubre los samplers estocásticos** porque hace falta poder inyectar el camino browniano; es una
  extensión pendiente del módulo de samplers, junto con el parámetro $\lambda$ que interpola entre PF-ODE y SDE reversa.
- Las métricas de acá viven **en el notebook**, no en el paquete. Cuando la evaluación de Fase 1 se especifique como módulo, el
  lugar natural de `w2_exacta`, `ksd2` y la discrepancia por componente es ahí, con su suite de tests — igual que el resto.
- Falta la **recuperación de parámetros** por EM sobre las muestras generadas, y la verosimilitud exacta por PF-ODE con la
  traza del jacobiano 2 × 2.